# Atelier de données — ce que tu peux fournir sans écrire le bambara

Une application web pour toi et tes amis, sur téléphone ou ordinateur. Rien
à installer de leur côté : un lien, et c'est parti.

| Onglet | Ce qu'on fait | À quoi ça sert |
|---|---|---|
| 🗣️ **Dire** | une phrase française s'affiche, on la dit en bambara comme on parle (mélange avec le français permis), à un ami, à un inconnu ou à une personne âgée, en plusieurs versions si on veut ; 300 phrases en 12 rubriques | **jeu de test maison** : compréhension depuis l'audio, cascade contre bout-en-bout, sans aucune transcription |
| 📖 **Lire** | lire à voix haute des phrases bambara déjà écrites | voix de synthèse et reconnaissance (audio + texte exact) |
| ⚖️ **Juger** | deux traductions, laquelle est la meilleure ? à l'aveugle | **évaluation humaine** des modèles, en plus du chrF++ |
| 📚 **Lexique** | argot et mots empruntés (*wai*, *yoh*…) et leur sens | normaliser l'entrée avant la traduction |
| 📊 **Bilan** | qui a fait quoi, minutes, préférences, export | suivre la collecte |

Chacun voit sa progression (au total et par rubrique) et reprend là où il
s'était arrêté.

**Mode d'emploi**

1. **Exécution → Modifier le type d'exécution → CPU.** L'atelier n'a pas
   besoin de GPU : garde ton quota pour l'entraînement.
2. **Exécution → Tout exécuter**, et autoriser l'accès à Drive.
3. La dernière cellule affiche un lien `https://….gradio.live` : l'ouvrir sur
   ton téléphone, l'envoyer à tes amis. Il marche tant que cette cellule
   tourne ; chaque relance donne un nouveau lien.
4. Tout est écrit sur ton Drive à chaque clic (`MyDrive/bambara/atelier`) :
   si la session se coupe, tout relancer ; chacun reprend où il en était.

Garde l'onglet Colab ouvert pendant la collecte : sans activité, Colab
gratuit finit par couper la session.

Le dépôt GitHub est public : les enregistrements restent sur ton Drive et
n'y vont jamais. Détails, conseils de prise de son et usage des données :
[`docs/ATELIER.md`](https://github.com/LOSTHADY/IA_Project-/blob/claude/chat-ia-bambara-dho8oa/docs/ATELIER.md).

## 1. Installation

In [ ]:
# Branche de travail. Une fois fusionnée, la branche par défaut suffit (sans -b).
BRANCH = "claude/chat-ia-bambara-dho8oa"

import os
os.chdir("/content")
if os.path.isdir("projet/.git"):
    !git -C projet pull -q   # relance : code mis à jour, rien à recloner
else:
    !git clone -q -b $BRANCH https://github.com/LOSTHADY/IA_Project- projet
%cd /content/projet
!pip install -q gradio soundfile librosa datasets

## 2. Dossier sur Drive

`ATELIER` reçoit tout : contributeurs, prises, audio, jugements, lexique.
`OUT` est le dossier des rapports de la phase 3, d'où viennent les
traductions à juger.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ATELIER = "/content/drive/MyDrive/bambara/atelier"
OUT = "/content/drive/MyDrive/bambara/resultats-phase3"
!mkdir -p "$ATELIER"

## 3. Phrases à lire

Une fois pour toutes : 400 phrases bambara tirées de Bayelemabaga, de
longueur moyenne, sans chiffres (qui se disent de plusieurs façons), sans
textes religieux, sans doublons.

In [ ]:
from pathlib import Path
if Path(ATELIER, "lecture.jsonl").exists():
    print("Phrases à lire : déjà prêtes.")
else:
    !python -m eval.atelier --racine "$ATELIER" lecture

### Campagne de lecture (des dizaines de lecteurs)

Pour une campagne de plusieurs dizaines d'heures (docs/CAMPAGNE.md), 400
phrases ne suffisent pas : `CAMPAGNE = True` ajoute environ 30 000 phrases,
22 000 de conversation (transcriptions relues d'AfVoices, RobotsMali) et
8 000 avec leur français (Bayelemabaga). Les phrases déjà là, et déjà lues,
sont gardées. L'onglet Lire les distribue ensuite : chaque lecteur dans son
ordre, trois lecteurs au plus par phrase tant qu'il en reste.

In [ ]:
CAMPAGNE = False
if CAMPAGNE and not Path(ATELIER, "lecture.avant-campagne.jsonl").exists():
    !python scripts/campagne.py liste --atelier "$ATELIER"
!python scripts/campagne.py bilan --atelier "$ATELIER"

## 4. Traductions à juger

Chaque lot oppose deux systèmes sur les mêmes phrases du test de Jeli-ASR ;
le côté de chacun (A ou B) est tiré au hasard à chaque phrase, et les
traductions identiques sont écartées. Le premier lot porte sur la question du
mémoire : la cascade contre le bout-en-bout, jugés sur ce qui a été compris
de l'audio.

Un lot est préparé une seule fois, dès que ses deux rapports existent dans
`OUT` : ceux de la v2 apparaîtront d'eux-mêmes après le second tour, à la
prochaine relance. Les jugements déjà faits sont gardés.

In [ ]:
# (nom du lot, système A, système B, sens) : les noms des rapports de la phase 3.
LOTS = [
    ("1-cascade-contre-bout-en-bout", "cascade Soloni, NLLB affiné", "bout-en-bout affiné", "bm2fr"),
    ("2-affinage-fr2bm", "NLLB zero-shot / Jeli-ASR", "NLLB affiné / Jeli-ASR", "fr2bm"),
    ("3-affinage-bm2fr", "NLLB zero-shot / Jeli-ASR", "NLLB affiné / Jeli-ASR", "bm2fr"),
    ("4-v1-contre-v2-fr2bm", "NLLB affiné / Jeli-ASR", "NLLB v2 / Jeli-ASR", "fr2bm"),
    ("5-v1-contre-v2-bm2fr", "NLLB affiné / Jeli-ASR", "NLLB v2 / Jeli-ASR", "bm2fr"),
]
PAIRES_PAR_LOT = 100

from pathlib import Path
from eval.atelier import preparer_lot, trouver_rapport

for nom, a, b, sens in LOTS:
    if Path(ATELIER, "lots", f"{nom}.jsonl").exists():
        print(f"{nom} : déjà prêt")
        continue
    rapports = [trouver_rapport(OUT, r) if Path(OUT).is_dir() else None for r in (a, b)]
    manquants = [r for r, p in zip((a, b), rapports) if p is None]
    if manquants:
        print(f"{nom} : en attente du rapport {manquants}")
        continue
    _, stats = preparer_lot(*rapports, sens, ATELIER, n=PAIRES_PAR_LOT, nom=nom)
    print(f"{nom} : {stats['retenues']} paires à juger ({stats['identiques']} traductions "
          f"identiques écartées sur {stats['communes']})")

## 5. L'atelier

Le lien public s'affiche ci-dessous (`Running on public URL`). La cellule ne
rend pas la main : l'arrêter (■) ferme l'atelier, sans rien perdre.

Le micro du navigateur ne marche qu'en `https` : utiliser le lien
`gradio.live`, y compris sur ton propre téléphone.

In [ ]:
!python -u app/atelier_app.py --share --racine "$ATELIER"

## 6. Après la collecte

Le bouton **Exporter** de l'onglet Bilan fait la même chose. Il écrit dans
`ATELIER` :

- `testset-dire.jsonl` : le jeu de test maison, évalué par le notebook de la
  phase 3 (§9b), cascade contre bout-en-bout ;
- `lecture-export.jsonl` : audio et texte exact, pour la synthèse vocale ;
- `lexique.tsv` : l'argot et les emprunts.

Arrêter d'abord la cellule précédente, puis exécuter celle-ci.

In [ ]:
!python -m eval.atelier --racine "$ATELIER" exporter